# Lesson 8.5: Logistic Regression: Predicting Yes or No

Predict pass or fail from study hours with logistic regression for the lesson's 20 students: coefficients, odds ratio, predicted probability and a confusion matrix.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the pass/fail data

Hours studied for 20 students and whether each passed (1) or failed (0). An outcome that is only yes or no needs logistic regression, not a straight line.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

hours = [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75,
         3, 3.25, 3.5, 4, 4.25, 4.5, 4.75, 5, 5.5, 6]
passed = [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1]
df = pd.DataFrame({'hours': hours, 'passed': passed})

print('students passed:', df.passed.sum(), 'of', len(df))

students passed: 10 of 20


## 2. Fit the logistic regression

logit models the log-odds of passing as a straight line in hours. The coefficients are on the log-odds scale: the slope is how much the log-odds rise for one extra hour.

In [2]:
model = smf.logit('passed ~ hours', data=df).fit(disp=0)

print(model.params.round(4))
print()
print(model.pvalues.round(4))
print(f'log-likelihood = {model.llf:.4f}')

Intercept   -4.2542
hours        1.4487
dtype: float64

Intercept    0.0213
hours        0.0164
dtype: float64
log-likelihood = -7.7461


## 3. Odds ratios and a predicted probability

Exponentiating a coefficient turns it into an odds ratio: the factor by which the odds of passing are multiplied for each extra hour. Putting a number of hours through the model gives the probability of passing.

In [3]:
odds_ratio = np.exp(model.params['hours'])
ci = np.exp(model.conf_int().loc['hours'])

print(f'odds ratio per hour = {odds_ratio:.2f}')
print(f'95% CI: ({ci[0]:.2f}, {ci[1]:.2f})')

p3 = model.predict(pd.DataFrame({'hours': [3]}))[0]
print(f'probability of passing after 3 hours = {p3:.3f}')

odds ratio per hour = 4.26
95% CI: (1.30, 13.90)
probability of passing after 3 hours = 0.523


## 4. Confusion matrix at a 0.5 cut-off

To turn probabilities into yes/no predictions, predict 'pass' when the probability is 0.5 or more. The matrix counts correct and wrong predictions in each group.

In [4]:
prob = model.predict(df)
pred = (prob >= 0.5).astype(int)

table = pd.crosstab(pred, df.passed, rownames=['predicted'], colnames=['actually passed'])
print(table)
print(f'accuracy = {(pred == df.passed).mean():.2f}')

actually passed  0  1
predicted            
0                8  2
1                2  8
accuracy = 0.80


## Try this

- Predict the probability of passing after 1 hour and after 5 hours, then check with model.predict.
- Change the cut-off from 0.5 to 0.3. Predict whether the number of false alarms (predicted pass, actually failed) goes up or down.

## The same idea in R, SPSS and Stata

**R**

```r
hours <- c(0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 4, 4.25, 4.5, 4.75, 5, 5.5, 6)
passed <- c(0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1)
fit <- glm(passed ~ hours, family = binomial)
summary(fit)
exp(coef(fit))
predict(fit, data.frame(hours = 3), type = "response")
table(predicted = fitted(fit) >= 0.5, actual = passed)
```

**SPSS** (Analyze > Regression > Binary Logistic (Dependent: passed, Covariate: hours; Options: CI for Exp(B)))

```
DATA LIST FREE / hours passed.
BEGIN DATA
0.5 0  0.75 0  1 0  1.25 0  1.5 0  1.75 1  2 0  2.25 0  2.5 0  2.75 1
3 0  3.25 1  3.5 1  4 0  4.25 1  4.5 1  4.75 1  5 1  5.5 1  6 1
END DATA.
LOGISTIC REGRESSION VARIABLES passed
  /METHOD=ENTER hours
  /PRINT=CI(95)
  /CRITERIA=CUT(0.5).
```

**Stata**

```
clear
set obs 20
generate hours = real(word("0.5 0.75 1 1.25 1.5 1.75 2 2.25 2.5 2.75 3 3.25 3.5 4 4.25 4.5 4.75 5 5.5 6", _n))
generate passed = real(word("0 0 0 0 0 1 0 0 0 1 0 1 1 0 1 1 1 1 1 1", _n))
logit passed hours
logistic passed hours
estat classification
margins, at(hours = 3)
```